# **🏠 Heritage Housing — Data Cleaning**

---

## 📋 1. Data Cleaning Summary

### Objectives

The objective of this notebook is to inspect, clean, and prepare the Heritage Housing dataset for further analysis and modelling.

The notebook will:

* Explore the structure and quality of the dataset.
* Identify and investigate missing values.
* Apply appropriate treatments to categorical and numerical missing data.
* Perform data quality checks.
* Save the cleaned dataset for use in subsequent notebooks.

### Input

* `outputs/datasets/collection/house_prices_records.csv`

### Output

* `outputs/datasets/cleaned/house_prices_cleaned.csv`

---

## ⚙️ 2. Set Up the Notebook

### Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues'

---

## 📥 3. Load Data

Import heritage housing historical data.

In [4]:
import pandas as pd

df = pd.read_csv("outputs/datasets/collection/house_prices_records.csv")

df.head(3)

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


---

## 🔍 4. Data Exploration

### 4.1 Pandas Profiling Report
For data cleaning, I will inspect the distribution and shape of variables containing missing values.

In [5]:
vars_with_missing_data = df.columns[df.isna().sum() > 0].to_list()
vars_with_missing_data

['2ndFlrSF',
 'BedroomAbvGr',
 'BsmtExposure',
 'BsmtFinType1',
 'EnclosedPorch',
 'GarageFinish',
 'GarageYrBlt',
 'LotFrontage',
 'MasVnrArea',
 'WoodDeckSF']

In [17]:
from ydata_profiling import ProfileReport
if vars_with_missing_data:
    profile = ProfileReport(df=df[vars_with_missing_data], minimal=True)
    profile.to_notebook_iframe()
else:
    print("There are no variables with missing data")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### 4.2 Interpretation of Pandas Profiling Report
* **24 variables** in total
  * **20 numerical variables**
  * **4 categorical variables**
    *  `BsmtExposure` – Basement exposure
    *  `BsmtFinType1` – Type of basement finish
    *  `GarageFinish` – Garage finish level
    *  `KitchenQual` – Kitchen quality
* `OverallQual` and `OverallCond` are numerical variables with an **ordinal interpretation**.
* `BedroomAbvGr` is also a numerical variable, but it represents a **count** rather than a continuous measurement.
* There are **missing values in several variables**.

### 4.3 Missing Values Overview

I will check the number of missing values in each variable and sort them from highest to lowest.

In [15]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': (missing / len(df) * 100).round(2)
})

missing_summary

,Missing Values,Percentage (%)
EnclosedPorch,1324,90.68
WoodDeckSF,1305,89.38
LotFrontage,259,17.74
BedroomAbvGr,99,6.78
2ndFlrSF,86,5.89
GarageYrBlt,81,5.55
MasVnrArea,8,0.55


## 🧹 5. Data Cleaning

### 5.1 Handle Missing Categorical Values

Of the 4 categorical variables, 3 contain missing values:

* GarageFinish — 235 missing values
* BsmtFinType1 — 145 missing values
* BsmtExposure — 38 missing values

The remaining categorical variable, `KitchenQual`, contains **no missing values** and requires no treatment.

#### 5.1.1 Investigate Missing Categorical Values
First, I will check whether the missing values in the three categorical variables represent the absence of the corresponding feature:

* **GarageFinish** — compare with `GarageArea`.
* **BsmtFinType1** — compare with `TotalBsmtSF`.
* **BsmtExposure** — compare with `TotalBsmtSF`.

A value of `0` in the related numerical variable indicates that the garage or basement is absent.

In [7]:
# Check whether missing categorical values represent absence of the feature

feature_checks = {
    'GarageFinish': 'GarageArea',
    'BsmtFinType1': 'TotalBsmtSF',
    'BsmtExposure': 'TotalBsmtSF'
}

for cat_col, num_col in feature_checks.items():
    missing = df.loc[df[cat_col].isna(), num_col]

    print(f"\n{cat_col} ({len(missing)} missing)")
    print(f"{num_col} = 0: {(missing == 0).sum()}")
    print(f"{num_col} > 0: {(missing > 0).sum()}")


GarageFinish (235 missing)
GarageArea = 0: 81
GarageArea > 0: 154

BsmtFinType1 (145 missing)
TotalBsmtSF = 0: 37
TotalBsmtSF > 0: 108

BsmtExposure (38 missing)
TotalBsmtSF = 0: 37
TotalBsmtSF > 0: 1


The results show that:

* **GarageFinish:** 81 missing values correspond to properties with no garage, while 154 occur where a garage exists.
* **BsmtFinType1:** 37 missing values correspond to properties with no basement, while 108 occur where a basement exists.
* **BsmtExposure:** 37 missing values correspond to properties with no basement, while only 1 occurs where a basement exists.


In [8]:
# Replace NaN with 'None' where the garage/basement is absent

df.loc[
    df['GarageFinish'].isna() & (df['GarageArea'] == 0),
    'GarageFinish'
] = 'None'

df.loc[
    df['BsmtFinType1'].isna() & (df['TotalBsmtSF'] == 0),
    'BsmtFinType1'
] = 'None'

df.loc[
    df['BsmtExposure'].isna() & (df['TotalBsmtSF'] == 0),
    'BsmtExposure'
] = 'None'

In [ ]:
# Check remaining missing values after replacing absent features with 'None'
df[['GarageFinish', 'BsmtFinType1', 'BsmtExposure']].isna().sum()

GarageFinish    154
BsmtFinType1    108
BsmtExposure      1
dtype: int64

The remaining missing values occur where the garage or basement exists, but the categorical information is unavailable. These values will be replaced with a `"Missing"` category using `CategoricalImputer`.

In [12]:
from feature_engine.imputation import CategoricalImputer

cat_imputer = CategoricalImputer(
    imputation_method='missing',
    fill_value='Missing',
    variables=[
        'GarageFinish',
        'BsmtFinType1',
        'BsmtExposure'
    ]
)

df = cat_imputer.fit_transform(df)

/home/cistudent/.local/lib/python3.12/site-packages/feature_engine/imputation/categorical.py:232: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(X[variable]):


Confirm that all missing categorical values have been resolved:

In [13]:
df[
    ['GarageFinish', 'BsmtFinType1', 'BsmtExposure']
].isna().sum()

GarageFinish    0
BsmtFinType1    0
BsmtExposure    0
dtype: int64

### 5.2 Handle Missing Numerical Values

Of the 20 numerical variables, 7 contain missing values:

* EnclosedPorch — 1,324 missing values
* WoodDeckSF — 1,305 missing values
* LotFrontage — 259 missing values
* BedroomAbvGr — 99 missing values
* 2ndFlrSF — 86 missing values
* GarageYrBlt — 81 missing values
* MasVnrArea — 8 missing values

The remaining 13 numerical variables contain no missing values and require no treatment.

* **Replace with `0`:** `EnclosedPorch`, `WoodDeckSF`, `2ndFlrSF`, and `MasVnrArea`, where missing values can reasonably represent the absence of the feature.
* **Replace with the median:** `LotFrontage`, `BedroomAbvGr`, and `GarageYrBlt`, where `0` would not be a realistic replacement.

The median is preferred for genuinely missing numerical values because it is less sensitive to extreme values than the mean.

In [9]:
# Fill missing values where NaN represents the absence of a feature
absence_cols = [
    'EnclosedPorch',
    'WoodDeckSF',
    '2ndFlrSF',
    'MasVnrArea'
]

df[absence_cols] = df[absence_cols].fillna(0)


# Fill genuinely missing numerical values with the median
median_cols = [
    'LotFrontage',
    'BedroomAbvGr',
    'GarageYrBlt'
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

### 5.3 Confirm Missing Values are Resolved

In [12]:
df.isnull().sum().sort_values(ascending=False)

All variables now report zero missing values so our cleanup has been successful!

---

## 🔎 6. Data Quality Checks

Before saving the cleaned dataset, perform final quality checks.

### 6.1 Check for Duplicate Records
No suplicate records have been found which is what we hoped for.

In [10]:
df.duplicated().sum()

### 6.2 Check Data Types

This confirms that numerical and categorical variables retain appropriate data types after cleaning.

In [11]:
df.dtypes

### 6.3 Check Dataset Shape

The number of rows has remained unchanged which is what we would have expected.

In [12]:
df.shape

### 6.4 Final Dataset Preview

The dataset is now cleaned while retaining its original categorical variables. Encoding and other feature transformations will be performed later when required for analysis or modelling.

In [13]:
df.head(3)

---

## 💾 6. 

---

## ✅ 7. Conclusions